In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
# Project root directory
project_dir = Path.cwd().parent

# Raw dataset path
raw_file = project_dir / "raw_data" / "retail_store_sales.csv"

# Load dataset
df_raw = pd.read_csv(raw_file)

print("Dataset loaded successfully!")
print("Project directory:", project_dir)
print("Shape:", df_raw.shape)

Dataset loaded successfully!
Project directory: F:\ABHISHEK\DATA SCIENCE\Mini Projects\retail-data-quality-eda
Shape: (12575, 11)


In [3]:

df = df_raw.copy()

print("Raw dataset shape:", df_raw.shape)
print("Working dataset shape:", df.shape)

print("Original data preserved:", df_raw.equals(df))

Raw dataset shape: (12575, 11)
Working dataset shape: (12575, 11)
Original data preserved: True


In [4]:
quality_report = pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2),
    "Unique_Values": df.nunique()
})

display(quality_report)

,Data_Type,Missing_Count,Missing_Percentage,Unique_Values
Transaction ID,object,0,0.00,12575
Customer ID,object,0,0.00,25
Category,object,0,0.00,8
Item,object,1213,9.65,200
Price Per Unit,float64,609,4.84,25
Quantity,float64,604,4.80,10
Total Spent,float64,604,4.80,227
Payment Method,object,0,0.00,3
Location,object,0,0.00,2
Transaction Date,object,0,0.00,1114


In [5]:
missing_pattern = df[
    df[["Item", "Price Per Unit", "Quantity", "Total Spent", "Discount Applied"]]
    .isna()
    .any(axis=1)
]

print("Rows with at least one missing value:", len(missing_pattern))

display(
    missing_pattern[
        ["Transaction ID", "Item", "Price Per Unit",
         "Quantity", "Total Spent", "Discount Applied"]
    ].head(10)
)

Rows with at least one missing value: 4996


,Transaction ID,Item,Price Per Unit,Quantity,Total Spent,Discount Applied
3,TXN_9458126,Item_16_BEV,27.5,9.0,247.5,NaN
5,TXN_7482416,NaN,NaN,10.0,200.0,NaN
7,TXN_1372952,NaN,33.5,NaN,NaN,True
11,TXN_5422631,NaN,NaN,8.0,52.0,True
14,TXN_2490363,Item_16_MILK,27.5,2.0,55.0,NaN
15,TXN_1809665,NaN,24.5,NaN,NaN,NaN
17,TXN_9634894,NaN,NaN,10.0,275.0,NaN
19,TXN_4206593,NaN,35.0,NaN,NaN,False
20,TXN_9939063,Item_7_BEV,14.0,9.0,126.0,NaN
21,TXN_8685338,NaN,NaN,3.0,105.0,NaN


In [6]:
missing_pattern = missing_pattern.copy()

missing_pattern["Missing_Count"] = missing_pattern[
    ["Item", "Price Per Unit", "Quantity", "Total Spent", "Discount Applied"]
].isna().sum(axis=1)

display(missing_pattern[["Transaction ID", "Missing_Count"]].head(10))

,Transaction ID,Missing_Count
3,TXN_9458126,1
5,TXN_7482416,3
7,TXN_1372952,3
11,TXN_5422631,2
14,TXN_2490363,1
15,TXN_1809665,4
17,TXN_9634894,3
19,TXN_4206593,3
20,TXN_9939063,1
21,TXN_8685338,3


In [7]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values("Missing_Count", ascending=False)

display(missing_summary)

,Missing_Count,Missing_Percentage
Discount Applied,4199,33.39
Item,1213,9.65
Price Per Unit,609,4.84
Quantity,604,4.80
Total Spent,604,4.80


In [8]:
df["Discount Applied"].value_counts(dropna=False)

Discount Applied
True     4219
NaN      4199
False    4157
Name: count, dtype: int64

In [9]:
df["Item"].value_counts(dropna=False).head(10)

Item
NaN             1213
Item_2_BEV       126
Item_25_FUR      113
Item_11_FUR      110
Item_16_MILK     109
Item_1_MILK      109
Item_5_FUR       107
Item_20_BUT      107
Item_14_FOOD     106
Item_11_MILK     106
Name: count, dtype: int64

In [10]:
df.loc[
    df["Item"].isna(),
    ["Transaction ID", "Category", "Price Per Unit", "Quantity", "Total Spent"]
].head(10)

,Transaction ID,Category,Price Per Unit,Quantity,Total Spent
5,TXN_7482416,Patisserie,NaN,10.0,200.0
7,TXN_1372952,Furniture,33.5,NaN,NaN
11,TXN_5422631,Milk Products,NaN,8.0,52.0
15,TXN_1809665,Beverages,24.5,NaN,NaN
17,TXN_9634894,Milk Products,NaN,10.0,275.0
19,TXN_4206593,Furniture,35.0,NaN,NaN
21,TXN_8685338,Milk Products,NaN,3.0,105.0
25,TXN_3481599,Furniture,39.5,NaN,NaN
32,TXN_1543244,Food,NaN,8.0,196.0
34,TXN_1621497,Patisserie,23.0,NaN,NaN


In [11]:
df["Item"] = df["Item"].fillna("Unavailable")

print("Missing Item values:", df["Item"].isna().sum())
print(df["Item"].value_counts().head())

Missing Item values: 0
Item
Unavailable     1213
Item_2_BEV       126
Item_25_FUR      113
Item_11_FUR      110
Item_16_MILK     109
Name: count, dtype: int64


In [12]:
print("Price Per Unit missing:", df["Price Per Unit"].isna().sum())
print("Quantity missing:", df["Quantity"].isna().sum())
print("Total Spent missing:", df["Total Spent"].isna().sum())

print(
    "Teeno mein se kisi ek ki value missing:",
    df[["Price Per Unit", "Quantity", "Total Spent"]]
    .isna().any(axis=1).sum()
)

Price Per Unit missing: 609
Quantity missing: 604
Total Spent missing: 604
Teeno mein se kisi ek ki value missing: 1213


In [13]:
# Missing Total Spent, but Price and Quantity available
recoverable_total = (
    df["Total Spent"].isna()
    & df["Price Per Unit"].notna()
    & df["Quantity"].notna()
)

print("Total Spent recover kiya ja sakta hai:", recoverable_total.sum())

Total Spent recover kiya ja sakta hai: 0


In [14]:
recoverable_price = (
    df["Price Per Unit"].isna()
    & df["Quantity"].notna()
    & df["Total Spent"].notna()
)

print("Price Per Unit recover ho sakta hai:", recoverable_price.sum())

Price Per Unit recover ho sakta hai: 609


In [15]:
recoverable_price = (
    df["Price Per Unit"].isna()
    & df["Quantity"].notna()
    & df["Total Spent"].notna()
    & df["Quantity"].ne(0)
)

df.loc[recoverable_price, "Price Per Unit"] = (
    df.loc[recoverable_price, "Total Spent"]
    / df.loc[recoverable_price, "Quantity"]
)

print("Remaining missing prices:", df["Price Per Unit"].isna().sum())

Remaining missing prices: 0


In [16]:
print("Missing Quantity:", df["Quantity"].isna().sum())
print("Missing Total Spent:", df["Total Spent"].isna().sum())

Missing Quantity: 604
Missing Total Spent: 604


In [17]:
print(
    "Quantity missing, but Total Spent available:",
    (df["Quantity"].isna() & df["Total Spent"].notna()).sum()
)

Quantity missing, but Total Spent available: 0


In [18]:
incomplete_transactions = df[
    df[["Quantity", "Total Spent"]].isna().any(axis=1)
]

print("Incomplete transactions:", len(incomplete_transactions))

Incomplete transactions: 604


In [19]:
recoverable_total = (
    df["Total Spent"].isna()
    & df["Price Per Unit"].notna()
    & df["Quantity"].notna()
)

print("Total Spent recover ho sakta hai:", recoverable_total.sum())

Total Spent recover ho sakta hai: 0


In [20]:
recoverable_quantity = (
    df["Quantity"].isna()
    & df["Price Per Unit"].notna()
    & df["Total Spent"].notna()
)

print("Quantity recover ho sakti hai:", recoverable_quantity.sum())

Quantity recover ho sakti hai: 0


In [21]:
print(
    "Quantity aur Total Spent dono missing:",
    (df["Quantity"].isna() & df["Total Spent"].isna()).sum()
)

Quantity aur Total Spent dono missing: 604


In [22]:
incomplete_transactions = df[
    df["Quantity"].isna() & df["Total Spent"].isna()
].copy()

print("Incomplete transactions:", len(incomplete_transactions))
print("Main dataset rows:", len(df))

Incomplete transactions: 604
Main dataset rows: 12575


In [23]:
print("Original missing prices:", df_raw["Price Per Unit"].isna().sum())
print("Current missing prices:", df["Price Per Unit"].isna().sum())

Original missing prices: 609
Current missing prices: 0


In [24]:
print("Zero prices:", (df["Price Per Unit"] == 0).sum())
print("Negative prices:", (df["Price Per Unit"] < 0).sum())
print("Missing prices:", df["Price Per Unit"].isna().sum())

Zero prices: 0
Negative prices: 0
Missing prices: 0


In [25]:
print("Total missing values:", df.isna().sum().sum())

Total missing values: 5407


In [26]:
df.columns

Index(['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit',
       'Quantity', 'Total Spent', 'Payment Method', 'Location',
       'Transaction Date', 'Discount Applied'],
      dtype='object')

In [27]:
df["Price Per Unit"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 12575 entries, 0 to 12574
Series name: Price Per Unit
Non-Null Count  Dtype  
--------------  -----  
12575 non-null  float64
dtypes: float64(1)
memory usage: 98.4 KB


In [28]:
print("Final dataset shape:", df.shape)
print("\nMissing values by column:")
print(df.isna().sum())

Final dataset shape: (12575, 11)

Missing values by column:
Transaction ID         0
Customer ID            0
Category               0
Item                   0
Price Per Unit         0
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


In [29]:
incomplete_transactions[
    ["Transaction ID", "Category", "Item", "Price Per Unit", "Quantity", "Total Spent"]
].head(10)

,Transaction ID,Category,Item,Price Per Unit,Quantity,Total Spent
7,TXN_1372952,Furniture,Unavailable,33.5,NaN,NaN
15,TXN_1809665,Beverages,Unavailable,24.5,NaN,NaN
19,TXN_4206593,Furniture,Unavailable,35.0,NaN,NaN
25,TXN_3481599,Furniture,Unavailable,39.5,NaN,NaN
34,TXN_1621497,Patisserie,Unavailable,23.0,NaN,NaN
83,TXN_4532214,Patisserie,Unavailable,15.5,NaN,NaN
112,TXN_5419965,Computers and electric accessories,Unavailable,20.0,NaN,NaN
114,TXN_8210621,Beverages,Unavailable,39.5,NaN,NaN
118,TXN_8586300,Computers and electric accessories,Unavailable,27.5,NaN,NaN
138,TXN_1154680,Furniture,Unavailable,35.0,NaN,NaN


In [30]:
df.groupby("Category")["Discount Applied"].value_counts(dropna=False)

Category                            Discount Applied
Beverages                           NaN                 551
                                    True                521
                                    False               495
Butchers                            True                543
                                    False               514
                                    NaN                 511
Computers and electric accessories  False               546
                                    True                535
                                    NaN                 477
Electric household essentials       NaN                 541
                                    False               528
                                    True                522
Food                                False               547
                                    NaN                 533
                                    True                508
Furniture                           NaN        

In [31]:
print(df.dtypes)

Transaction ID       object
Customer ID          object
Category             object
Item                 object
Price Per Unit      float64
Quantity            float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object
Discount Applied     object
dtype: object


In [32]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])

print(df["Transaction Date"].dtype)

datetime64[ns]


In [33]:
print(df["Discount Applied"].dtype)

object


In [34]:
df["Discount Applied"] = df["Discount Applied"].astype("boolean")

print(df["Discount Applied"].dtype)

boolean


In [35]:
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())

Dataset shape: (12575, 11)

Data types:
Transaction ID              object
Customer ID                 object
Category                    object
Item                        object
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
Discount Applied           boolean
dtype: object

Missing values:
Transaction ID         0
Customer ID            0
Category               0
Item                   0
Price Per Unit         0
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


In [36]:
originally_missing_price = df_raw["Price Per Unit"].isna()

print(
    "Filled prices with missing Quantity or Total Spent:",
    (
        originally_missing_price
        & (
            df[["Quantity", "Total Spent"]].isna().any(axis=1)
        )
    ).sum()
)

Filled prices with missing Quantity or Total Spent: 0


In [37]:
cleaning_report = pd.DataFrame({
    "Original Missing": df_raw.isna().sum(),
    "Remaining Missing": df.isna().sum()
})

cleaning_report["Values Handled"] = (
    cleaning_report["Original Missing"]
    - cleaning_report["Remaining Missing"]
)

display(cleaning_report)

,Original Missing,Remaining Missing,Values Handled
Transaction ID,0,0,0
Customer ID,0,0,0
Category,0,0,0
Item,1213,0,1213
Price Per Unit,609,0,609
Quantity,604,604,0
Total Spent,604,604,0
Payment Method,0,0,0
Location,0,0,0
Transaction Date,0,0,0


In [38]:
processed_dir = project_dir / "processed_data"
processed_dir.mkdir(parents=True, exist_ok=True)

cleaned_file = processed_dir / "retail_store_sales_cleaned.csv"

df.to_csv(cleaned_file, index=False)

print("Cleaned dataset saved successfully!")
print("File location:", cleaned_file)

Cleaned dataset saved successfully!
File location: F:\ABHISHEK\DATA SCIENCE\Mini Projects\retail-data-quality-eda\processed_data\retail_store_sales_cleaned.csv


In [39]:
df_cleaned = pd.read_csv(cleaned_file)

print("Saved dataset shape:", df_cleaned.shape)
print("File exists:", cleaned_file.exists())

Saved dataset shape: (12575, 11)
File exists: True
